In [13]:
from typing import Any
from mdp import RLAlgorithm, MDP, FlackyTramMDP
from collections import defaultdict
import numpy as np
from game import Game, State, HalvingState, HalvingGame, HalvingState
import random

In [14]:
# Tabular TD Learning Agent
# Estimates V_pi(s) while using 1-step lookahead over the known MDP to act.
class TDLearning(RLAlgorithm):

    def __init__(self, 
                mdp: MDP, 
                exploration_policy, 
                epsilon: float = 0.2, 
                discount: float = 1.0, 
                learning_rate: float = 0.1):
        self.mdp = mdp
        self.exploration_policy = exploration_policy
        self.epsilon = epsilon
        self.discount = discount
        self.learning_rate = learning_rate
        self.V = defaultdict(float)

    def get_action(self, state: Any) -> Any:
        if np.random.random() < self.epsilon:
            return self.exploration_policy(state)
        return self.pi(state)

    def pi(self, state: Any) -> Any:
        successors = self.mdp.successors(state)
        q_values = defaultdict(float)
        for step in successors:
            action, next_state, prob, reward = step.action, step.state, step.prob, step.reward
            # Compute Q(s, a) = sum_s' P(s'|s, a) * [R + gamma * V(s')]
            q_values[action] += prob * (reward + self.discount * self.V[next_state])
        return max(q_values.keys(), key=lambda a: q_values[a])

    def incorporate_feedback(self, state: Any, action: Any, reward: float, next_state: Any, is_end: bool):
        # TD update step: V(s) <- V(s) + alpha * (target - V(s)).
        if is_end:
            target = reward
        else:
            target = reward + self.discount * self.V[next_state]
        self.V[state] += self.learning_rate * (target - self.V[state])

# --- Verification & Simulation Run ---
def walk_tram_policy(num_locs: int, state: int) -> str:
    """Default exploratory policy: walk when low, tram when halfway."""
    return "walk" if state < (num_locs // 2) else "tram"

mdp = FlackyTramMDP(num_locs=6, failure_prob=0.1)
agent = TDLearning(
    mdp=mdp,
    exploration_policy=lambda s: walk_tram_policy(6, s),
    epsilon=0.2,
    discount=1.0,
    learning_rate=0.05
)

# Simulate multiple episodes to train V(s)
num_episodes = 500
for ep in range(num_episodes):
    s = mdp.start_state()
    while not mdp.is_end(s):
        a = agent.get_action(s)
        successors = [step for step in mdp.successors(s) if step.action == a]
        # Sample successor based on probabilities
        probs = [succ.prob for succ in successors]
        chosen_idx = np.random.choice(len(successors), p=probs)
        succ = successors[chosen_idx]

        agent.incorporate_feedback(
            state=s, action=a, reward=succ.reward, next_state=succ.state, is_end=mdp.is_end(succ.state)
        )
        s = succ.state

print("Learned Value Function V(s):")
for loc in range(1, 6):
    print(f"State{loc}: V ={agent.V[loc]:.3f}, Greedy Action ={agent.pi(loc)}")


Learned Value Function V(s):
State1: V =-4.152, Greedy Action =walk
State2: V =-3.141, Greedy Action =walk
State3: V =-2.415, Greedy Action =tram
State4: V =-5.000, Greedy Action =walk
State5: V =0.000, Greedy Action =walk


In [15]:
# TD-learning with linear function approximation for 2-player zero-sum games.
# V_w(s) = w . phi(s)

class LinearGameTDLearning:

    def __init__(self, feature_extractor, num_features: int, lr: float = 0.1, discount: float = 1.0):
        self.feature_extractor = feature_extractor
        self.lr = lr
        self.discount = discount
        self.weights = np.zeros(num_features)

    def get_value(self, state: Any) -> float:
        return float(np.dot(self.weights, self.feature_extractor(state)))

    def incorporate_feedback(self, state: Any, reward: float, next_state: Any, is_end: bool):
        # TD update step: V(s) <- V(s) + alpha * (target - V(s)).
        if is_end:
            target = reward
        else:
            target = reward + self.discount * self.get_value(next_state)

        self.weights += self.lr * (target - self.get_value(state)) * self.feature_extractor(state)

    def get_action(self, game: Game, state: Any, epsilon: float = 0.1) -> Any:
        successors = game.successors(state)
        if np.random.random() < epsilon:
            return random.choice(list(successors.keys()))
        values = {action: self.get_value(next_state) if not game.is_end(next_state) else game.utility(next_state) for action, next_state in successors.items()}
        player = game.player(state)
        if player == 'agent':
            return max(values.keys(), key = lambda a: values[a])
        else:
            return min(values.keys(), key=lambda a: values[a])

In [22]:
# Indicator feature representation
N_MAX = 15
def extract_indicator_features(state: HalvingState):
    n = state.n
    player = state.player
    phi = np.zeros(2 * (N_MAX + 1))
    if 0 <= n <= N_MAX:
        offset = 0 if player == "agent" else (N_MAX + 1)
        phi[offset + n] = 1.0
    return phi

# Self-Play Training Loop
game = HalvingGame(n=15)
num_features = 2 * (N_MAX + 1)
td_agent = LinearGameTDLearning(feature_extractor=extract_indicator_features, num_features=num_features, lr=0.05)

episodes = 5000
for _ in range(episodes):
    # 随机从 1..N_MAX 开始，保证每个状态都有足够的访问次数
    start_n = random.randint(1, N_MAX)
    game = HalvingGame(start_n)
    s = game.start_state()
    
    while not game.is_end(s):
        a = td_agent.get_action(game, s, epsilon=0.15)
        next_s = game.successors(s)[a]
        
        is_terminal = game.is_end(next_s)
        reward = game.utility(next_s) if is_terminal else 0.0
        
        td_agent.incorporate_feedback(s, reward, next_s, is_terminal)
        s = next_s

# Evaluate Learned State Values from Agent's perspective
print("Learned State Values (N -> Value from Agent's turn):")
for num in range(1, 16):
    val = td_agent.get_value(HalvingState(num, "agent"))
    print(f"N ={num:2d} | V(s) ={val:+.3f}")

Learned State Values (N -> Value from Agent's turn):
N = 1 | V(s) =+1.000
N = 2 | V(s) =-1.000
N = 3 | V(s) =+0.796
N = 4 | V(s) =+0.640
N = 5 | V(s) =+0.897
N = 6 | V(s) =-0.840
N = 7 | V(s) =+0.744
N = 8 | V(s) =-0.847
N = 9 | V(s) =+0.424
N =10 | V(s) =-0.635
N =11 | V(s) =+0.587
N =12 | V(s) =+0.783
N =13 | V(s) =+0.692
N =14 | V(s) =-0.750
N =15 | V(s) =+0.512
